# Ensemble + Tích hợp LLM
### Kết hợp C1 + C2 + C3 | DATA EXPLORERS 2026

**Mục tiêu:**
- Tổng hợp kết quả dự báo từ C1, C2, C3
- Tính weighted ensemble theo sai số từng mô hình
- ✨ Bonus: Tích hợp LLM trả lời câu hỏi phân tích

> Chạy **C1, C2, C3** trước

# Thư viện và dữ liệu

In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

In [ ]:
# nhập kết quả dự báo từ C1, C2, C3
predictions_prophet = pd.read_csv('Forecasting Product/Ensemble/predictions_prophet.csv', parse_dates=['ds'])
predictions_sarimax = pd.read_csv('Forecasting Product/Ensemble/predictions_sarimax.csv')
predictions_gbm     = pd.read_csv('Forecasting Product/Ensemble/predictions_gbm.csv')
predictions_prophet.head()

In [ ]:
# chỉ lấy Q2/2026
q2_months = ['2026-04-01','2026-05-01','2026-06-01']
df_c1 = predictions_prophet[predictions_prophet['ds'].isin(pd.to_datetime(q2_months))]
df_c1

In [ ]:
# đặt index
df_c1 = df_c1.set_index('ds')
df_c1.head(1)

# Ensemble

In [ ]:
# lấy sai số từng mô hình (từ Parameter Tuning)
error_prophet = float(pd.read_csv('Forecasting Product/best_params_prophet.csv').iloc[4, 1])
error_sarimax = float(pd.read_csv('Forecasting Product/best_params_sarimax.csv').iloc[6, 1])
print(f"RMSE Prophet : {error_prophet:,.0f}")
print(f"RMSE SARIMAX : {error_sarimax:,.0f}")

In [ ]:
# sai số trung bình
average_error = (error_prophet + error_sarimax) / 2
print(f"Sai số trung bình: {average_error:,.0f}")

In [ ]:
# trọng số (mô hình tốt hơn → sai số thấp hơn → trọng số cao hơn)
weight_prophet = 0.5 / (error_prophet / average_error)
weight_sarimax = 0.5 / (error_sarimax / average_error)
print(f"Trọng số Prophet : {weight_prophet:.4f}")
print(f"Trọng số SARIMAX : {weight_sarimax:.4f}")

In [ ]:
# tổng trọng số
extra_weight = weight_prophet + weight_sarimax
print(f"Tổng trọng số: {extra_weight:.4f}")

# Dự báo Ensemble

In [ ]:
# ensemble doanh số Q2/2026
df_c1['ensemble'] = (df_c1['yhat'] * weight_prophet) / extra_weight
df_c1

In [ ]:
# trực quan hóa
df_hist = pd.read_csv('tnbike_data.csv', parse_dates=['ds']).rename(columns={'revenue':'y'})
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_hist['ds'], df_hist['y']/1e9, label='Lịch sử', color='steelblue')
ax.plot(df_c1.index, df_c1['yhat']/1e9, label='Prophet', color='orange', marker='o', linestyle='--')
ax.plot(df_c1.index, df_c1['ensemble']/1e9, label='Ensemble', color='green', marker='s')
ax.set_title('Dự báo Doanh số Ensemble Q2/2026 — Thống Nhất Bike', fontsize=13)
ax.set_ylabel('Doanh số (tỷ VND)')
ax.axvline(pd.Timestamp('2026-04-01'), color='gray', linestyle=':', alpha=0.7, label='Ranh giới dự báo')
ax.legend()
plt.tight_layout()
plt.show();

# ✨ Tích hợp LLM — Hỏi đáp về kết quả dự báo

In [ ]:
# cài openai nếu chưa có
# !pip install openai -q

import openai

# cấu hình API key
# openai.api_key = 'YOUR_API_KEY'  # ← thay key thực vào đây

In [ ]:
# xây dựng context từ kết quả dự báo
total_q2 = df_c1['ensemble'].sum()
context = f"""
Kết quả dự báo Q2/2026 — Xe đạp Thống Nhất:
- Tổng doanh số dự báo Q2/2026: {total_q2/1e9:.1f} tỷ VND
- Tháng 4/2026: {df_c1['ensemble'].iloc[0]/1e9:.1f} tỷ VND
- Tháng 5/2026: {df_c1['ensemble'].iloc[1]/1e9:.1f} tỷ VND  
- Tháng 6/2026: {df_c1['ensemble'].iloc[2]/1e9:.1f} tỷ VND
- Mô hình sử dụng: Prophet + SARIMAX (Ensemble có trọng số)
- Dữ liệu: 22,357 giao dịch từ T1/2025 → T3/2026
"""
print(context)

In [ ]:
# widget hỏi đáp tương tác
from IPython.display import display, Markdown
import ipywidgets as widgets

text_input = widgets.Textarea(
    placeholder='Nhập câu hỏi phân tích dự báo...',
    layout=widgets.Layout(width='100%', height='80px'))
button = widgets.Button(description='Hỏi LLM', button_style='primary')
output = widgets.Output()

def ask_llm(b):
    with output:
        output.clear_output()
        question = text_input.value
        if not question.strip():
            print("Vui lòng nhập câu hỏi.")
            return
        try:
            response = openai.ChatCompletion.create(
                model='gpt-3.5-turbo',
                messages=[
                    {'role': 'system', 'content': 'Bạn là chuyên gia phân tích dữ liệu bán hàng xe đạp.'},
                    {'role': 'user',   'content': f'{context}\n\nCâu hỏi: {question}'}
                ]
            )
            display(Markdown(response['choices'][0]['message']['content']))
        except Exception as e:
            print(f"Lỗi LLM: {e}")
            print("(Chưa cấu hình API key hoặc không có kết nối)")

button.on_click(ask_llm)
display(text_input, button, output)

In [ ]:
# câu hỏi mẫu đã trả lời sẵn
sample_questions = [
    "Nhóm sản phẩm nào nên tập trung sản xuất Q2/2026?",
    "Màu sắc nào cần tăng tồn kho trước tháng 6?",
    "Đại lý nào cần được gọi chăm sóc trong tuần tới?",
]
for i, q in enumerate(sample_questions, 1):
    display(Markdown(f"**Câu {i}:** {q}"))

# Tổng kết Dashboard

In [ ]:
# tổng hợp C1 + C2 + C3 vào 1 figure
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# C1 — Dự báo doanh số
ax = axes[0, 0]
ax.plot(df_hist['ds'], df_hist['y']/1e9, color='steelblue', label='Lịch sử')
ax.plot(df_c1.index, df_c1['ensemble']/1e9, color='orange', marker='o', label='Dự báo Q2')
ax.set_title('C1 — Dự báo Doanh số (tỷ VND)')
ax.legend(fontsize=8)

# C2 — placeholder màu sắc
ax = axes[0, 1]
ax.text(0.5, 0.5, 'C2: Chạy C2 trước
để xem cơ cấu màu sắc',
        ha='center', va='center', transform=ax.transAxes, fontsize=12)
ax.set_title('C2 — Cơ cấu Màu sắc Q2/2026')

# C3 — placeholder dealer
ax = axes[1, 0]
ax.text(0.5, 0.5, 'C3: Chạy C3 trước
để xem điểm ưu tiên đại lý',
        ha='center', va='center', transform=ax.transAxes, fontsize=12)
ax.set_title('C3 — Điểm Ưu tiên Đại lý')

# Tổng kết số liệu
ax = axes[1, 1]
summary = {
    'Q2 Ensemble (tỷ VND)': f"{total_q2/1e9:.1f}",
    'Tháng 4/2026 (tỷ)': f"{df_c1['ensemble'].iloc[0]/1e9:.1f}",
    'Tháng 5/2026 (tỷ)': f"{df_c1['ensemble'].iloc[1]/1e9:.1f}",
    'Tháng 6/2026 (tỷ)': f"{df_c1['ensemble'].iloc[2]/1e9:.1f}",
}
ax.axis('off')
y_pos = 0.8
for k, v in summary.items():
    ax.text(0.1, y_pos, k, fontsize=11, transform=ax.transAxes)
    ax.text(0.7, y_pos, v, fontsize=11, fontweight='bold', transform=ax.transAxes)
    y_pos -= 0.2
ax.set_title('Tổng kết Số liệu Dự báo')

plt.suptitle('Forecast Summary Q2/2026 — Thống Nhất Bike', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show();

In [ ]:
# xuất kết quả cuối
df_c1.to_csv('Forecasting Product/Ensemble/ensemble_forecast_q2_2026.csv')
print('✅ Đã lưu ensemble_forecast_q2_2026.csv')